In [25]:
from pathlib import Path
import numpy as np
import pandas as pd

In [26]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
project_root = Path.cwd().parent
metadata_dir = project_root / "data" / "metadata"

In [27]:
inventory = pd.read_csv(
    metadata_dir / "data_inventory.csv"
)
inventory = inventory[inventory["is_valid"]].copy()
print(inventory.shape)

(2036, 23)


In [28]:
exp_svm = pd.read_csv(
    metadata_dir / "selected_svm_experimental_speakers.csv"
)

val_enrolled = pd.read_csv(
    metadata_dir / "selected_validation_enrolled_speakers.csv"
)

val_unknown = pd.read_csv(
    metadata_dir / "selected_validation_unknown_speakers.csv"
)

test_enrolled = pd.read_csv(
    metadata_dir / "selected_test_enrolled_speakers.csv"
)

test_unknown = pd.read_csv(
    metadata_dir / "selected_test_unknown_speakers.csv"
)

In [29]:
def split_audio(df, seed=42):
    df = df.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    enrollment = df.iloc[:5].copy()
    enrollment["dataset_type"] = "SVM_CLOSED_SET"
    enrollment["split_role"] = "ENROLLMENT"
    enrollment["split_name"] = "svm_closed_set_enrollment"

    train = df.iloc[5:15].copy()
    train["dataset_type"] = "SVM_CLOSED_SET"
    train["split_role"] = "TRAIN"
    train["split_name"] = "svm_closed_set_train"

    validation = df.iloc[15:20].copy()
    validation["dataset_type"] = "SVM_CLOSED_SET"
    validation["split_role"] = "VALIDATION"
    validation["split_name"] = "svm_closed_set_validation"

    test = df.iloc[20:25].copy()
    test["dataset_type"] = "SVM_CLOSED_SET"
    test["split_role"] = "TEST"
    test["split_name"] = "svm_closed_set_test"

    return enrollment, train, validation, test

### Sinh Closed-set SVM

In [30]:
enrollment_list = []
train_list = []
validation_list = []
test_list = []

for speaker in exp_svm["speaker_id"]:

    speaker_df = inventory[
        inventory["speaker_id"] == speaker
    ]

    enroll, train, val, test = split_audio(
        speaker_df,
        RANDOM_SEED
    )

    enrollment_list.append(enroll)
    train_list.append(train)
    validation_list.append(val)
    test_list.append(test)

In [31]:
svm_enrollment = pd.concat(
    enrollment_list,
    ignore_index=True
)

svm_train = pd.concat(
    train_list,
    ignore_index=True
)

svm_validation = pd.concat(
    validation_list,
    ignore_index=True
)

svm_test = pd.concat(
    test_list,
    ignore_index=True)

### Check

In [32]:
print(len(svm_enrollment))
print(len(svm_train))
print(len(svm_validation))
print(len(svm_test))

50
100
50
50


### Cosine Validation

In [33]:
val_enrollment = []
val_query = []

for speaker in val_enrolled["speaker_id"]:

    speaker_df = inventory[
        inventory["speaker_id"] == speaker
    ]

    speaker_df = speaker_df.sample(
        frac=1,
        random_state=RANDOM_SEED
    )

    enroll = speaker_df.iloc[:5].copy()
    enroll["dataset_type"] = "COSINE_VALIDATION"
    enroll["split_role"] = "ENROLLMENT"
    enroll["split_name"] = "cosine_validation_enrollment"

    query = speaker_df.iloc[5:10].copy()
    query["dataset_type"] = "COSINE_VALIDATION"
    query["split_role"] = "QUERY"
    query["split_name"] = "cosine_validation_query"

    val_enrollment.append(enroll)
    val_query.append(query)

val_enrollment = pd.concat(
    val_enrollment,
    ignore_index=True
)

val_query = pd.concat(
    val_query,
    ignore_index=True
)

### Validation Unknown

In [34]:
unknown_list = []

for speaker in val_unknown["speaker_id"]:

    speaker_df = inventory[
        inventory["speaker_id"] == speaker
    ].copy()

    speaker_df["dataset_type"] = "COSINE_VALIDATION"
    speaker_df["split_role"] = "UNKNOWN"
    speaker_df["split_name"] = "cosine_validation_unknown"

    unknown_list.append(speaker_df)

cosine_validation_unknown = pd.concat(
    unknown_list,
    ignore_index=True
)

### Cosine Test

In [35]:
test_enrollment_list = []
test_query_list = []

for speaker in test_enrolled["speaker_id"]:

    speaker_df = inventory[
        inventory["speaker_id"] == speaker
    ]

    speaker_df = speaker_df.sample(
        frac=1,
        random_state=RANDOM_SEED
    )

    enroll = speaker_df.iloc[:5].copy()
    enroll["dataset_type"] = "COSINE_TEST"
    enroll["split_role"] = "ENROLLMENT"
    enroll["split_name"] = "cosine_test_enrollment"

    query = speaker_df.iloc[5:10].copy()
    query["dataset_type"] = "COSINE_TEST"
    query["split_role"] = "QUERY"
    query["split_name"] = "cosine_test_query"

    test_enrollment_list.append(enroll)
    test_query_list.append(query)

cosine_test_enrollment = pd.concat(
    test_enrollment_list,
    ignore_index=True
)

cosine_test_query = pd.concat(
    test_query_list,
    ignore_index=True
)

### Test Unknown

In [36]:
unknown_list = []

for speaker in test_unknown["speaker_id"]:

    speaker_df = inventory[
        inventory["speaker_id"] == speaker
    ].copy()

    speaker_df["dataset_type"] = "COSINE_TEST"
    speaker_df["split_role"] = "UNKNOWN"
    speaker_df["split_name"] = "cosine_test_unknown"

    unknown_list.append(speaker_df)

cosine_test_unknown = pd.concat(
    unknown_list,
    ignore_index=True
)

### Leakage Check

In [37]:
def check_overlap(df1, df2):
    return len(
        set(df1.audio_id)
        &
        set(df2.audio_id)
    )

In [38]:
print(check_overlap(svm_enrollment, svm_train))
print(check_overlap(svm_train, svm_validation))
print(check_overlap(svm_train, svm_test))
print(check_overlap(val_enrollment, val_query))
print(check_overlap(cosine_test_enrollment, cosine_test_query))

0
0
0
0
0


In [39]:
def check_speaker_overlap(df1, df2):
    return set(df1["speaker_id"]) & set(df2["speaker_id"])

checks = {
    "SVM vs Validation":
        check_speaker_overlap(
            exp_svm,
            pd.concat([val_enrolled, val_unknown])
        ),

    "SVM vs Test":
        check_speaker_overlap(
            exp_svm,
            pd.concat([test_enrolled, test_unknown])
        ),

    "Validation vs Test":
        check_speaker_overlap(
            pd.concat([val_enrolled, val_unknown]),
            pd.concat([test_enrolled, test_unknown])
        ),

    "Validation enrolled vs Validation unknown":
        check_speaker_overlap(
            val_enrolled,
            val_unknown
        ),

    "Test enrolled vs Test unknown":
        check_speaker_overlap(
            test_enrolled,
            test_unknown
        )
}

for name, overlap in checks.items():
    print(f"{name}: {overlap}") 

SVM vs Validation: set()
SVM vs Test: set()
Validation vs Test: set()
Validation enrolled vs Validation unknown: set()
Test enrolled vs Test unknown: set()


In [40]:
import hashlib

def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()

split_dfs = [
    svm_enrollment,
    svm_train,
    svm_validation,
    svm_test,
    val_enrollment,
    val_query,
    cosine_validation_unknown,
    cosine_test_enrollment,
    cosine_test_query,
    cosine_test_unknown
]


for df in split_dfs:

    df["checksum"] = df["audio_path"].apply(
        lambda x:
        calculate_sha256(
            project_root
            / "data"
            / "audio"
            / x
        )
    )

In [41]:
for df in split_dfs:
    print(df["checksum"].isna().sum())

0
0
0
0
0
0
0
0
0
0


In [42]:
svm_enrollment.to_csv(
    metadata_dir / "svm_closed_set_enrollment.csv",
    index=False,
    encoding="utf-8-sig"
)

svm_train.to_csv(
    metadata_dir / "svm_closed_set_train.csv",
    index=False,
    encoding="utf-8-sig"
)

svm_validation.to_csv(
    metadata_dir / "svm_closed_set_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

svm_test.to_csv(
    metadata_dir / "svm_closed_set_test.csv",
    index=False,
    encoding="utf-8-sig"
)

val_enrollment.to_csv(
    metadata_dir / "cosine_validation_enrollment.csv",
    index=False,
    encoding="utf-8-sig"
)

val_query.to_csv(
    metadata_dir / "cosine_validation_query.csv",
    index=False,
    encoding="utf-8-sig"
)

cosine_validation_unknown.to_csv(
    metadata_dir / "cosine_validation_unknown.csv",
    index=False,
    encoding="utf-8-sig"
)

cosine_test_enrollment.to_csv(
    metadata_dir / "cosine_test_enrollment.csv",
    index=False,
    encoding="utf-8-sig"
)

cosine_test_query.to_csv(
    metadata_dir / "cosine_test_query.csv",
    index=False,
    encoding="utf-8-sig"
)

cosine_test_unknown.to_csv(
    metadata_dir / "cosine_test_unknown.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Split completed.")

Split completed.


In [43]:
summary = pd.DataFrame({
    "file": [
        "svm_closed_set_enrollment",
        "svm_closed_set_train",
        "svm_closed_set_validation",
        "svm_closed_set_test",
        "cosine_validation_enrollment",
        "cosine_validation_query",
        "cosine_validation_unknown",
        "cosine_test_enrollment",
        "cosine_test_query",
        "cosine_test_unknown"
    ],
    "num_audio": [
        len(svm_enrollment),
        len(svm_train),
        len(svm_validation),
        len(svm_test),
        len(val_enrollment),
        len(val_query),
        len(cosine_validation_unknown),
        len(cosine_test_enrollment),
        len(cosine_test_query),
        len(cosine_test_unknown)
    ],
    "num_speaker": [
        svm_enrollment["speaker_id"].nunique(),
        svm_train["speaker_id"].nunique(),
        svm_validation["speaker_id"].nunique(),
        svm_test["speaker_id"].nunique(),
        val_enrollment["speaker_id"].nunique(),
        val_query["speaker_id"].nunique(),
        cosine_validation_unknown["speaker_id"].nunique(),
        cosine_test_enrollment["speaker_id"].nunique(),
        cosine_test_query["speaker_id"].nunique(),
        cosine_test_unknown["speaker_id"].nunique()
    ]
})

summary

,file,num_audio,num_speaker
0,svm_closed_set_enrollment,50,10
1,svm_closed_set_train,100,10
2,svm_closed_set_validation,50,10
3,svm_closed_set_test,50,10
4,cosine_validation_enrollment,10,2
5,cosine_validation_query,10,2
6,cosine_validation_unknown,144,2
7,cosine_test_enrollment,10,2
8,cosine_test_query,10,2
9,cosine_test_unknown,116,2


In [44]:
from datetime import datetime
import json

manifest = {
    "version": "v1",
    "created_at": datetime.now().isoformat(),
    "random_seed": RANDOM_SEED,

    "speaker_groups": {
        "svm_experimental": len(exp_svm),
        "validation_enrolled": len(val_enrolled),
        "validation_unknown": len(val_unknown),
        "test_enrolled": len(test_enrolled),
        "test_unknown": len(test_unknown)
    },

    "audio_counts": {
        "svm_closed_set_enrollment": len(svm_enrollment),
        "svm_closed_set_train": len(svm_train),
        "svm_closed_set_validation": len(svm_validation),
        "svm_closed_set_test": len(svm_test),

        "cosine_validation_enrollment": len(val_enrollment),
        "cosine_validation_query": len(val_query),
        "cosine_validation_unknown": len(cosine_validation_unknown),

        "cosine_test_enrollment": len(cosine_test_enrollment),
        "cosine_test_query": len(cosine_test_query),
        "cosine_test_unknown": len(cosine_test_unknown)
    }
}

In [45]:
with open(
    metadata_dir / "split_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved split_manifest.json")

Saved split_manifest.json
